# SDP pipeline analisis de log

Cada SDP guardar eventos y metricas en la localización del almacenamiento definido para el pipeline. Con esta tabla podemos ver que está pasando y la calidad de datos procesados.

Puede consultas las tablas en forma directa con SQL y enviar alertas. 
Este notebook extrae y analiza metricas de espectativas para construir KPI.

Publicar el log de eventos en el UC:
1.	Settings > “Advanced settings” > presione el botón “Edit advanced settings”.
2.	Active la cada de chequeo “Publish evento log to unity catalog”.
3.	Suministre el nombre de catálogo, esquema para log de eventos y nombre de la tabla, por ejemplo, event_log_MiPipeline.

## Consultar SDP eventos en UC

In [0]:
%sql
SELECT * FROM medallion_dev.bronze.event_log_medallon_pipeline

id sequence origin timestamp message level maturity_level error details event_type dfb2b9a0-34f5-11f1-82c6-0a7b497d1746 List(List(execution, 1775836298135020), 1775836604988001) List(AWS, us-east-2, 7474651374019400, null, ecb32ef9-0676-4ad4-adb7-2e81cdc6424a, WORKSPACE, Medallion_Pipeline, null, f0d07d76-a2dd-477f-928b-fdb573b6e2f2, null, null, null, null, null, null, f0d07d76-a2dd-477f-928b-fdb573b6e2f2, null, null, null, null, null, null, null, null, null, null, null, null) 2026-04-10T15:56:44.986Z User hugo14.gonzalez@gmail.com started an update. INFO STABLE null {"user_action":{"action":"START","user_name":"hugo14.gonzalez@gmail.com","user_id":77345568936916,"request":{"start_request":{"full_refresh":false,"validate_only":true,"explore_only":false,"development":true}}}} user_action dfb8d420-34f5-11f1-82c6-0a7b497d1746 List(List(execution, 1775836298135021), 1775836605028001) List(AWS, us-east-2, 7474651374019400, null, ecb32ef9-0676-4ad4-adb7-2e81cdc6424a, WORKSPACE, Medallion_Pipeline, null, f0d07d76-a2dd-477f-928b-fdb573b6e2f2, null, null, null, null, null, null, f0d07d76-a2dd-477f-928b-fdb573b6e2f2, null, null, null, null, null, null, null, null, null, null, null, null) 2026-04-10T15:56:44.995Z Update f0d07d started by USER_ACTION. INFO STABLE null {"create_update":{"cause":"USER_ACTION","config":{"id":"ecb32ef9-0676-4ad4-adb7-2e81cdc6424a","pipeline_type":"WORKSPACE","name":"Medallion_Pipeline","configuration":{"pipelines.acfsProcessorVersion":"2","pipelines.allowClearingTableComment":"true","pipelines.allowForwardReferencesInSinkRegistration":"true","pipelines.allowPythonDecoratedDatasetAccessDunderName":"true","pipelines.alterExistingMvSt":"true","pipelines.alterViewsInHMS.enabled":"true","pipelines.alterableMetadataBehavior":"preserve","pipelines.analysis.allowSqlFlowWithConfParallelResolution":"true","pipelines.analysis.enableImmediateFlowFailureEventEmission":"false","pipelines.analysis.maybeResolveFlowsParallely":"true","pipelines.analysis.maybeResolvePureSQLPipelinesParallely":"true","pipelines.analysis.pruneUnresolvedFromJsonDependents":"true","pipelines.applyChanges.Type2IgnoreNullPersistentStateChange":"true","pipelines.applyChanges.Type2OmitNullEntriesInVersionMap.Read":"true","pipelines.applyChanges.Type2OmitNullEntriesInVersionMap.Write":"false","pipelines.applyChanges.applyChangesMultiflowIgnoreNullOffFix.enabled":"true","pipelines.applyChanges.applyChangesMultiflowLatestDeleteVersion.enabled":"true","pipelines.applyChanges.dynamicFilePruning.enabled":"true","pipelines.applyChanges.dynamicPartitionPruning.enabled":"true","pipelines.applyChanges.useAfterImageStatusColumn.enabled":"false","pipelines.applyChangesFromSnapshot.analyzeLambdaSourceWithoutPipelineContext":"true","pipelines.applyChangesFromSnapshot.castSnapshotToTargetSchema":"true","pipelines.applyChangesFromSnapshot.stateStoreFormat":"proto","pipelines.assignSqlStatesToBuiltInPythonErrors":"true","pipelines.asyncServerInit":"true","pipelines.attachEnzymeExpectationsUsingTransformDF":"true","pipelines.attachExpectationWithFunctionResult":"true","pipelines.attachExpectationsUsingTransformDF":"true","pipelines.autoCdc.logCaseDifferences":"false","pipelines.autoCdcColumnNormalizationRefactor":"true","pipelines.blockDuplicatedUseOfFromJsonSchemaLocationKey":"true","pipelines.brickstore.checkpointFPIPercentageForStreaming":"5","pipelines.brickstore.enableDirectToStorageSseCEncryption":"false","pipelines.brickstore.enableInitialLoadTableSizeCheck":"true","pipelines.brickstore.enableSyncedTableTimestampWithTz":"true","pipelines.brickstore.lockTimeoutMillis":"1200000","pipelines.bypassDltAnalysisForForeachBatchSql.enabled":"true","pipelines.capturePreExecutionEventLogTable":"true","pipelines.cdc.enableCdcSnapshotProgress":"true","pipelines.cdc.enableGatewayErrorPropagation":"true","pipelines.cdc.skipCdcSnapshotFlowAfterCommitted":"true","pipelines.cdc.stagingDataGCDryRunMode":"false","pipelines.cdc.stagingDataRetentionDays":"25","pipelines.cdcApplier.cd

## Analizar estructura de la tabla event log

La columna `details` contiene metadatos de cada evento. Los campos dependen del tipo de evento:
* `user_action` Acciones de usuario como crear pipeline
* `flow_definition` Cuando despliega o modifica pipeline y tiene lineage, schema y plan de ejecución.
  * `output_dataset` y `input_datasets` - output table/view and its upstream table(s)/view(s)
  * `flow_type` - whether this is a complete or append flow
  * `explain_text` - the Spark explain plan
* `flow_progress` Events occur when a data flow starts running or finishes processing a batch of data
  * `metrics` - currently contains `num_output_rows`
  * `data_quality` - contains an array of the results of the data quality rules for this particular dataset
    * `dropped_records`
    * `expectations`
      * `name`, `dataset`, `passed_records`, `failed_records`

In [0]:
%sql
SELECT
  details:flow_definition.output_dataset,
  details:flow_definition.input_datasets,
  details:flow_definition.flow_type,
  details:flow_definition.schema,
  details:flow_definition
FROM medallion_dev.bronze.event_log_medallon_pipeline
WHERE details:flow_definition IS NOT NULL
ORDER BY timestamp
LIMIT 10

output_dataset,input_datasets,flow_type,schema,flow_definition
medallion_dev.bronze.churn_event_bronze,null,APPEND,"[{""name"":""user_id"",""path"":[""user_id""],""data_type"":""STRING""},{""name"":""event_id"",""path"":[""event_id""],""data_type"":""STRING""},{""name"":""platform"",""path"":[""platform""],""data_type"":""STRING""},{""name"":""date"",""path"":[""date""],""data_type"":""STRING""},{""name"":""action"",""path"":[""action""],""data_type"":""STRING""},{""name"":""session_id"",""path"":[""session_id""],""data_type"":""STRING""},{""name"":""url"",""path"":[""url""],""data_type"":""STRING""},{""name"":""_rescued_data"",""path"":[""_rescued_data""],""data_type"":""STRING""},{""name"":""metadata"",""path"":[""metadata""],""data_type"":""STRING""},{""name"":""ingest_datetime"",""path"":[""ingest_datetime""],""data_type"":""TIMESTAMP""}]","{""output_dataset"":""medallion_dev.bronze.churn_event_bronze"",""explain_text"":""'Project [unresolvedstarwithcolumns(ingest_datetime, 'current_timestamp(), None)]"",""schema_json"":""{\""type\"":\""struct\"",\""fields\"":[{\""name\"":\""user_id\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""event_id\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""platform\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""date\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""action\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""session_id\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""url\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""_rescued_data\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""metadata\"",\""type\"":\""string\"",\""nullable\"":false,\""metadata\"":{}},{\""name\"":\""ingest_datetime\"",\""type\"":\""timestamp\"",\""nullable\"":false,\""metadata\"":{}}]}"",""schema"":[{""name"":""user_id"",""path"":[""user_id""],""data_type"":""STRING""},{""name"":""event_id"",""path"":[""event_id""],""data_type"":""STRING""},{""name"":""platform"",""path"":[""platform""],""data_type"":""STRING""},{""name"":""date"",""path"":[""date""],""data_type"":""STRING""},{""name"":""action"",""path"":[""action""],""data_type"":""STRING""},{""name"":""session_id"",""path"":[""session_id""],""data_type"":""STRING""},{""name"":""url"",""path"":[""url""],""data_type"":""STRING""},{""name"":""_rescued_data"",""path"":[""_rescued_data""],""data_type"":""STRING""},{""name"":""metadata"",""path"":[""metadata""],""data_type"":""STRING""},{""name"":""ingest_datetime"",""path"":[""ingest_datetime""],""data_type"":""TIMESTAMP""}],""flow_type"":""APPEND"",""comment"":""Bronze layer: Raw customer churn events ingested from CSV files using Auto Loader"",""spark_conf"":[{""key"":""spark.pipelines.flow.name"",""value"":""medallion_dev.bronze.churn_event_bronze""},{""key"":""spark.pipelines.flow.isResetting"",""value"":""false""}],""language"":""PYTHON"",""notebook_path"":""/Proyectos_Dev/databricks-medallion/Medallion_Pipeline/transformations/bronze.py"",""once"":false}"
medallion_dev.bronze.churn_event_bronze,null,APPEND,"[{""name"":""user_id"",""path"":[""user_id""],""data_type"":""STRING""},{""name"":""event_id"",""path"":[""event_id""],""data_type"":""STRING""},{""name"":""platform"",""path"":[""platform""],""data_type"":""STRING""},{""name"":""date"",""path"":[""date""],""data_type"":""STRING""},{""name"":""action"",""path"":[""action""],""data_type"":""STRING""},{""name"":""session_id"",""path"":[""session_id""],""data_type"":""STRING""},{""name"":""url"",""path"":[""url""],""data_type"":""STRING""},{""name"":""_rescued_data"",""path"":[""_rescued_data""],""data_type"":""STRING""},{""name"":""ingest_file_name"",""path"":[""ingest_file_name""],""data_type"":""STRING""},{""name"":""ingest_datetime"",""path"":[""ingest_datetime""],""data_type"":""TIMESTAMP""}]","{""

In [0]:
%sql
SELECT
  id,
  expectations.dataset,
  expectations.name,
  expectations.failed_records,
  expectations.passed_records
FROM(
  SELECT 
    id,
    timestamp,
    details:flow_progress.metrics,
    details:flow_progress.data_quality.dropped_records,
    explode(from_json(details:flow_progress:data_quality:expectations
             ,schema_of_json("[{'name':'str', 'dataset':'str', 'passed_records':42, 'failed_records':42}]"))) expectations
  FROM medallion_dev.bronze.event_log_medallon_pipeline
  WHERE details:flow_progress.metrics IS NOT NULL) data_quality

id,dataset,name,failed_records,passed_records
8ed8ad40-34f6-11f1-993c-9e5aeb727ea0,medallion_dev.bronze.churn_order_bronze,correct_schema,0,178061
eae21a90-3523-11f1-879f-d2aabccdfa11,medallion_dev.silver.churn_user_silver,id_valid,0,68879
f3196010-3523-11f1-879f-d2aabccdfa11,v_churn_order_silver_staging,user_id_valid,3517,174544
f3196010-3523-11f1-879f-d2aabccdfa11,v_churn_order_silver_staging,order_id_valid,3549,174512
df5c3070-348d-11f1-a9db-42142c6c3586,medallion_dev.bronze.churn_user_bronze,correct_schema,0,68879


## Eso es todo! Las metricas de calidad están listas! 

La tabla puede ser consultada usando DBSQL. Abra el <a dbdemos-dashboard-id="sdp-quality-stat" href='/sql/dashboardsv3/01f118a0ddea16c09a3d5954c9c49066' target="_blank">Dashboard Log SDP calidad de datos</a>

Inicio [Ir a tras a README.md]($../README.md)